# [JAX·TPU 선택 심화] 04 · 직접 촬영한 이미지로 바꾸기
**목표:** Pillow로 사진을 읽어 NumPy 데이터로 만들고 회수한 파인튜닝 체크포인트로 추론합니다.

사진과 체크포인트를 아직 준비하지 않았다면 해당 단계는 안내만 표시합니다. 원격 작업은 이 노트북에서 자동 시작하지 않습니다.

기본 HF·PyTorch 과정은 `hf_colab_gpu/notebooks`에 있습니다. 이 심화 과정은 프로젝트 최상위에서 `bash scripts/setup.sh --with-jax`로 준비합니다.

## 라이브러리를 직접 불러오기
가상환경은 라이브러리 버전을 구분하는 공간입니다. 아래 `import`가 이 노트북에서 실제로 사용하는 라이브러리입니다.

| 가져오는 이름 | 설치할 패키지 | 하는 일 |
|---|---|---|
| `numpy` | `numpy` | 이미지 배열, 라벨, `.npz` 파일 |
| `PIL.Image` | `Pillow` | 이미지 읽기와 크기 조절 |
| `matplotlib.pyplot` | `matplotlib` | 이미지와 그래프 표시 |
| `IPython` | `ipykernel`과 함께 설치 | 노트북 안에 그래프 표시 |

`pathlib`, `os`, `sys`, `json`, `hashlib`, `importlib.metadata`는 Python 표준 라이브러리이므로 따로 설치하지 않습니다. 필요한 추가 라이브러리는 사용하는 셀에서 직접 불러옵니다.

In [ ]:
from pathlib import Path
import os
import sys
import json
import hashlib
from importlib.metadata import version

candidates = [Path.cwd(), *Path.cwd().parents, Path("/content/vision-ai")]
ROOT = next((path for path in candidates if (path / ".vision-lab-root").is_file()), None)
if ROOT is None:
    raise RuntimeError(".vision-lab-root가 있는 수업 폴더에서 열거나 Colab에 실습 파일을 먼저 업로드하세요.")
os.chdir(ROOT)
os.environ.setdefault("MPLCONFIGDIR", str(ROOT / ".cache/matplotlib"))
print("Project:", ROOT)
print("Python:", sys.executable)

In [ ]:
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from IPython import get_ipython

ipython = get_ipython()
if ipython is not None:
    ipython.run_line_magic("matplotlib", "inline")
plt.rcParams.update({"figure.figsize": (9, 4), "font.size": 11,
                     "axes.spines.top": False, "axes.spines.right": False})
for package in ("numpy", "Pillow", "matplotlib", "ipykernel"):
    print(f"{package}: {version(package)}")

## 1. 촬영과 분할
`custom_images/train/cup`, `custom_images/validation/cup`, `custom_images/test/cup`처럼 분할마다 같은 클래스 폴더를 만드세요. 분류할 클래스는 2개 이상이어야 합니다.

같은 영상의 비슷한 프레임을 섞지 말고 촬영 날짜·배경·조명 단위로 분리합니다. 아래 코드는 완전히 동일한 이미지를 찾지만 비슷한 연속 프레임까지 자동으로 걸러내지는 않습니다.

In [ ]:
CUSTOM_INPUT = ROOT / "custom_images"
CUSTOM_DATA = ROOT / "data/my-workbench"
PREPARE_CUSTOM = False
IMAGE_PATH = CUSTOM_INPUT / "test/cup/example.jpg"
CHECKPOINT = ROOT / "results/gpu/finetuned_checkpoint.npz"
print("Input exists:", CUSTOM_INPUT.is_dir())
print("Checkpoint exists:", CHECKPOINT.is_file())

### 저장된 데이터 읽기와 무결성 확인
`np.load(..., allow_pickle=False)`로 이미지·라벨·ID를 읽습니다. SHA256과 분할별 ID를 검사해 다른 데이터가 섞이거나 손상된 경우 중단합니다. 이 함수의 본문도 아래에 모두 표시합니다.

In [ ]:
def digest(path):
    hasher = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            hasher.update(block)
    return hasher.hexdigest()

In [ ]:
def load_prepared(path):
    path = Path(path)
    manifest = json.loads((path / "manifest.json").read_text(encoding="utf-8"))
    classes = manifest["classes"]
    if len(classes) < 2 or len(set(classes)) != len(classes):
        raise ValueError("클래스 목록이 잘못됐습니다.")
    splits, all_ids = {}, set()
    for name in ("train", "validation", "test"):
        record = manifest["splits"][name]
        if record["file"] != f"{name}.npz":
            raise ValueError("데이터 파일 경로가 예상 형식과 다릅니다.")
        file = path / record["file"]
        if digest(file) != record["sha256"]:
            raise ValueError(f"{name} 데이터 체크섬 불일치")
        with np.load(file, allow_pickle=False) as a:
            images, labels, ids = a["images"], a["labels"], a["ids"].tolist()
        if images.dtype != np.uint8 or images.ndim != 4 or images.shape[-1] != 3:
            raise ValueError("images는 uint8 NHWC RGB여야 합니다.")
        if len(images) != len(labels) or len(ids) != len(labels) or len(labels) != record["count"]:
            raise ValueError("이미지·라벨·식별자 개수가 다릅니다.")
        if labels.ndim != 1 or not np.issubdtype(labels.dtype, np.integer) or len(labels) == 0 or labels.min() < 0 or labels.max() >= len(classes):
            raise ValueError("라벨 범위가 잘못됐습니다.")
        if len(set(ids)) != len(ids) or all_ids.intersection(ids):
            raise ValueError("분할 간 이미지 ID 중복")
        all_ids.update(ids)
        splits[name] = {"images": images, "labels": labels, "ids": ids}
    identity = {k: manifest[k] for k in ("classes", "splits", "seed", "preprocess")}
    if hashlib.sha256(json.dumps(identity, sort_keys=True).encode()).hexdigest() != manifest["dataset_sha256"]:
        raise ValueError("데이터 manifest 지문 불일치")
    return splits, manifest

### 분할을 NPZ와 manifest로 저장하기
이미지 배열은 압축한 `.npz`, 출처·클래스·파일 지문은 `manifest.json`으로 저장합니다. 같은 픽셀이 두 분할에 있으면 데이터 누출을 막기 위해 중단합니다.

In [ ]:
PREPROCESS = {"size": 224, "mean": [0.5] * 3, "std": [0.5] * 3,
              "resize": "bilinear", "layout": "NHWC"}

In [ ]:
def save_prepared(output, splits, classes, source, seed):
    output = Path(output)
    output.mkdir(parents=True, exist_ok=True)
    # Exact duplicate pixels across splits are excluded before this point.
    seen = {}
    for name, split in splits.items():
        for image in split["images"]:
            key = hashlib.sha256(image.tobytes()).hexdigest()
            if key in seen and seen[key] != name:
                raise ValueError(f"분할 간 중복 이미지: {seen[key]} / {name}")
            seen[key] = name
    records = {}
    for name, split in splits.items():
        path = output / f"{name}.npz"
        with path.with_suffix(".tmp").open("wb") as stream:
            np.savez_compressed(stream, images=split["images"], labels=split["labels"], ids=np.asarray(split["ids"], dtype=str))
        path.with_suffix(".tmp").replace(path)
        records[name] = {"file": path.name, "sha256": digest(path), "count": len(split["labels"]),
                         "per_class": {c: int(np.sum(split["labels"] == i)) for i, c in enumerate(classes)}}
    identity = {"classes": classes, "splits": records, "seed": seed, "preprocess": PREPROCESS}
    manifest = {"schema_version": 1, "dataset_name": source["name"], **identity, "source": source,
                "dataset_sha256": hashlib.sha256(json.dumps(identity, sort_keys=True).encode()).hexdigest()}
    (output / "manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    return manifest

## 2. 폴더와 클래스 확인
사진을 넣은 뒤 `PREPARE_CUSTOM=True`로 바꾸세요. 새 출력 폴더에만 저장합니다. 기존 데이터를 다시 보려면 `False`로 두면 됩니다.

In [ ]:
custom_splits = custom_manifest = None
if PREPARE_CUSTOM:
    if not (CUSTOM_INPUT / "train").is_dir():
        raise FileNotFoundError("custom_images/train/<클래스> 폴더가 필요합니다.")
    if CUSTOM_DATA.exists() and any(CUSTOM_DATA.iterdir()):
        raise FileExistsError("기존 데이터를 덮어쓰지 않습니다. 새 CUSTOM_DATA를 지정하세요.")
    custom_classes = sorted(path.name for path in (CUSTOM_INPUT / "train").iterdir() if path.is_dir())
    if len(custom_classes) < 2:
        raise ValueError("클래스가 2개 이상 필요합니다.")
    for split_name in ("train", "validation", "test"):
        folder = CUSTOM_INPUT / split_name
        if not folder.is_dir() or sorted(path.name for path in folder.iterdir() if path.is_dir()) != custom_classes:
            raise ValueError(f"{split_name}의 클래스 폴더가 train과 다릅니다.")
    print("Classes:", custom_classes)
else:
    print("사진을 준비했다면 PREPARE_CUSTOM=True로 바꾸세요.")

## 3. Pillow로 사진을 읽고 중복 확인
읽은 RGB 픽셀의 SHA256으로 중복 사진을 확인합니다. 서로 다른 크기로 저장한 복사본도 잡을 수 있도록 아래 저장 단계에서 224×224 픽셀을 다시 검사합니다.

In [ ]:
if PREPARE_CUSTOM:
    custom_splits, seen_pixels = {}, {}
    for split_name in ("train", "validation", "test"):
        images, labels, image_ids = [], [], []
        for label, class_name in enumerate(custom_classes):
            files = sorted(path for path in (CUSTOM_INPUT / split_name / class_name).iterdir()
                           if path.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"})
            if not files:
                raise ValueError(f"이미지가 없습니다: {split_name}/{class_name}")
            for path in files:
                with Image.open(path) as image:
                    rgb = image.convert("RGB")
                    fingerprint = hashlib.sha256(str(rgb.size).encode() + rgb.tobytes()).hexdigest()
                    if fingerprint in seen_pixels:
                        raise ValueError(f"중복 이미지: {seen_pixels[fingerprint]} / {path}")
                    seen_pixels[fingerprint] = str(path)
                    images.append(np.asarray(rgb.resize((224, 224), Image.Resampling.BILINEAR)))
                labels.append(label)
                image_ids.append(path.relative_to(CUSTOM_INPUT).as_posix())
        custom_splits[split_name] = {"images": np.stack(images),
                                     "labels": np.asarray(labels, dtype=np.int64), "ids": image_ids}

## 4. 직접 저장하고 다시 읽기

In [ ]:
if PREPARE_CUSTOM:
    custom_manifest = save_prepared(CUSTOM_DATA, custom_splits, custom_classes,
                                   {"name": "custom_workbench", "input_layout": "train,validation,test/class/image"}, 0)
if (CUSTOM_DATA / "manifest.json").is_file():
    custom_splits, custom_manifest = load_prepared(CUSTOM_DATA)
    print("Classes:", custom_manifest["classes"])
    print("Counts:", {name: len(split["labels"]) for name, split in custom_splits.items()})
else:
    print("준비한 개인 데이터가 없어 저장·읽기를 생략합니다.")

## 5. 학습 사진 확인

In [ ]:
if custom_manifest is not None:
    fig, axes = plt.subplots(1, len(custom_manifest["classes"]),
                             figsize=(3 * len(custom_manifest["classes"]), 3), squeeze=False)
    for label, name in enumerate(custom_manifest["classes"]):
        index = int(np.flatnonzero(custom_splits["train"]["labels"] == label)[0])
        axes[0, label].imshow(custom_splits["train"]["images"][index])
        axes[0, label].set_title(name)
        axes[0, label].axis("off")
    fig.suptitle("Your training images · inspect class and background clues")
    fig.tight_layout()
    plt.show()
else:
    print("개인 사진 갤러리를 생략합니다.")

## 6. 새 데이터로 추가 학습
`CUSTOM_DATA`에 만든 `train.npz`, `validation.npz`, `test.npz`, `manifest.json` 네 파일을 Colab CLI의 `upload` 명령으로 하나씩 올립니다. 업로드 대상은 각각 `/content/vision-ai/data/prepared/train.npz`처럼 원격의 `data/prepared` 폴더로 유지합니다. 따라서 `02_gpu_finetuning.ipynb`와 `03_tpu_and_compare.ipynb`의 `DATA_PATH`를 바꾸지 않아도 됩니다. 학습 코드는 manifest에 저장된 클래스 이름과 개수를 읽습니다.

Colab CLI로 노트북과 이 데이터를 업로드한 뒤 각 장치에서 셀을 실행하세요. 두 장치에서는 같은 학습 설정을 사용합니다. CLI의 생성·업로드·실행·다운로드·종료 명령은 `docs/direct-colab-cli.md`에 있습니다. 다운로드한 GPU 체크포인트는 `results/gpu/finetuned_checkpoint.npz`에 둡니다. 기존 실험 결과를 보존하려면 다운로드 전에 별도 폴더로 옮기세요. 원격 실행을 선택하면 직접 찍은 이미지가 본인의 Colab 런타임으로 전송됩니다.

아래 단계는 결과를 회수한 뒤 CPU에서 한 장을 추론하는 코드입니다. 경로가 없으면 연산을 생략합니다.

### JAX와 실행 장치 확인
`jax`는 자동 미분과 컴파일을, `jax.numpy`는 배열 연산을 제공합니다. 설치 패키지는 CPU에서 `jax`, GPU에서 `jax[cuda12]`, TPU에서 `jax[tpu]`입니다. 같은 `import`를 사용하지만 실행 환경에 맞는 패키지가 필요합니다.

GPU·TPU를 요청했는데 장치가 없으면 오류로 중단합니다. CPU로 몰래 바꾸지 않습니다. 강사용 CPU 검증 때만 `VISION_DEVICE=cpu`를 명시할 수 있으며 출력에도 CPU로 기록됩니다.

In [ ]:
import jax
import jax.numpy as jnp

REQUESTED_DEVICE = os.environ.get("VISION_DEVICE", 'cpu')
if REQUESTED_DEVICE not in {"cpu", "gpu", "tpu"}:
    raise ValueError("VISION_DEVICE는 cpu, gpu, tpu 중 하나여야 합니다.")
try:
    devices = jax.devices(REQUESTED_DEVICE)
except RuntimeError as error:
    raise RuntimeError(f"{REQUESTED_DEVICE.upper()}를 찾지 못했습니다. 해당 Colab 런타임에서 실행하세요.") from error
if not devices or devices[0].platform != REQUESTED_DEVICE:
    raise RuntimeError("요청한 장치가 없습니다. Codespaces 자체에는 Colab GPU·TPU가 연결되지 않습니다.")
device = devices[0]
device_info = {"requested": REQUESTED_DEVICE, "platform": device.platform,
               "device_kind": device.device_kind, "available_count": len(devices),
               "used_count": 1, "jax_version": jax.__version__}
print("Device:", device_info)

### 모델 계산 1 · 행렬 곱과 정규화
`jnp.matmul`이 가중치와 입력을 곱합니다. LayerNorm은 마지막 축의 평균과 분산으로 값을 정규화합니다.

In [ ]:
def dense(params, prefix, x):
    return jnp.matmul(x, params[prefix + ".weight"].T,
                      precision=jax.lax.Precision.HIGHEST) + params[prefix + ".bias"]

def layer_norm(params, prefix, x, epsilon):
    centered = x - jnp.mean(x, axis=-1, keepdims=True)
    variance = jnp.mean(centered * centered, axis=-1, keepdims=True)
    return centered * jax.lax.rsqrt(variance + epsilon) * params[prefix + ".weight"] + params[prefix + ".bias"]

### 모델 계산 2 · Attention과 Transformer 블록
Query·Key의 내적으로 토큰 사이의 점수를 계산하고 `jax.nn.softmax`로 가중치를 만듭니다. 잔차 연결과 GELU까지 아래 함수에서 확인할 수 있습니다.

In [ ]:
def transformer_block(params, index, x, config):
    prefix = f"vit.encoder.layer.{index}"
    normalized = layer_norm(params, prefix + ".layernorm_before", x, config["layer_norm_eps"])
    batch, tokens, hidden = normalized.shape
    heads = config["num_attention_heads"]
    head_dim = hidden // heads

    def project(name):
        value = dense(params, prefix + ".attention.attention." + name, normalized)
        return value.reshape(batch, tokens, heads, head_dim).transpose(0, 2, 1, 3)

    query, key, value = (project(name) for name in ("query", "key", "value"))
    scores = jnp.matmul(query, key.swapaxes(-1, -2), precision=jax.lax.Precision.HIGHEST)
    probabilities = jax.nn.softmax(scores / jnp.sqrt(jnp.float32(head_dim)), axis=-1)
    context = jnp.matmul(probabilities, value, precision=jax.lax.Precision.HIGHEST)
    context = context.transpose(0, 2, 1, 3).reshape(batch, tokens, hidden)
    x = x + dense(params, prefix + ".attention.output.dense", context)
    normalized = layer_norm(params, prefix + ".layernorm_after", x, config["layer_norm_eps"])
    intermediate = jax.nn.gelu(dense(params, prefix + ".intermediate.dense", normalized), approximate=False)
    return x + dense(params, prefix + ".output.dense", intermediate)

### 모델 계산 3 · 이미지를 패치 토큰으로 바꾸기
224×224 이미지를 16×16 패치로 바꾼 뒤 첫 11개 블록을 통과시킵니다. 이 부분은 추가 학습에서 고정하므로 출력을 캐시할 수 있습니다.

In [ ]:
def prefix_tokens(params, images, config):
    """Patch embedding plus blocks 0..10; this output is safe to cache."""
    if images.ndim != 4 or images.shape[1:] != (224, 224, 3):
        raise ValueError("Expected preprocessed NHWC images [N, 224, 224, 3]")
    x = jax.lax.conv_general_dilated(
        images.astype(jnp.float32),
        params["vit.embeddings.patch_embeddings.projection.weight"].transpose(2, 3, 1, 0),
        window_strides=(config["patch_size"], config["patch_size"]),
        padding="VALID", dimension_numbers=("NHWC", "HWIO", "NHWC"),
        precision=jax.lax.Precision.HIGHEST,
    ) + params["vit.embeddings.patch_embeddings.projection.bias"]
    x = x.reshape(x.shape[0], -1, config["hidden_size"])
    cls = jnp.broadcast_to(params["vit.embeddings.cls_token"], (x.shape[0], 1, config["hidden_size"]))
    x = jnp.concatenate([cls, x], axis=1) + params["vit.embeddings.position_embeddings"]
    for index in range(config["num_hidden_layers"] - 1):
        x = transformer_block(params, index, x, config)
    return x

### 모델 계산 4 · 마지막 블록과 분류기
마지막 블록의 CLS 토큰으로 분류합니다. `original_logits`는 앞서 정의한 함수들을 순서대로 호출해 원본 1,000개 클래스 점수를 계산합니다.

In [ ]:
def tail_features(params, tokens, config):
    x = transformer_block(params, config["num_hidden_layers"] - 1, tokens, config)
    return layer_norm(params, "vit.layernorm", x, config["layer_norm_eps"])[:, 0]

def tail_logits(params, tokens, config):
    return dense(params, "classifier", tail_features(params, tokens, config))

def original_logits(params, images, config):
    return tail_logits(params, prefix_tokens(params, images, config), config)

### 학습할 가중치와 파일 지문
아래 함수는 마지막 블록·LayerNorm·분류기의 이름을 고르고 가중치가 바뀌었는지 확인합니다. `hashlib`의 SHA256은 파일과 배열의 지문을 만드는 표준 라이브러리 함수입니다.

In [ ]:
def is_tail_parameter(name, config):
    return (name.startswith(f"vit.encoder.layer.{config['num_hidden_layers'] - 1}.")
            or name.startswith("vit.layernorm.") or name.startswith("classifier."))

def extract_tail(params, config):
    return {key: value for key, value in params.items() if is_tail_parameter(key, config)}

def initialize_head(class_count, hidden_size, seed=42, device=None):
    if class_count < 2:
        raise ValueError("Fine-tuning requires at least two classes")
    weight = 0.02 * jax.random.normal(jax.random.PRNGKey(seed), (class_count, hidden_size))
    return {"classifier.weight": jax.device_put(weight, device),
            "classifier.bias": jax.device_put(jnp.zeros(class_count, jnp.float32), device)}

def parameter_digest(params):
    digest = hashlib.sha256()
    for key in sorted(params):
        value = np.ascontiguousarray(jax.device_get(params[key]))
        digest.update(key.encode("utf-8"))
        digest.update(str(value.shape).encode("ascii"))
        digest.update(value.dtype.str.encode("ascii"))
        digest.update(value.tobytes())
    return digest.hexdigest()

def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

### 체크포인트 저장과 복원
`np.savez_compressed`와 `np.load`로 갱신한 마지막 블록·분류기를 저장하고 읽습니다. 복원할 때 원본 모델 revision과 각 배열 크기를 검사합니다.

In [ ]:
def save_checkpoint(path, tail, metadata):
    """Store only the adapted final block, norm and head; source stays unchanged."""
    arrays = {key: np.asarray(jax.device_get(value)) for key, value in tail.items()}
    arrays["__metadata__"] = np.array(json.dumps(metadata, ensure_ascii=False))
    np.savez_compressed(path, **arrays)

def load_checkpoint(path, source_params, config, device=None):
    with np.load(path, allow_pickle=False) as archive:
        metadata = json.loads(str(archive["__metadata__"]))
        if metadata.get("model_revision") != MODEL_REVISION:
            raise ValueError("Checkpoint revision does not match the pinned source")
        expected = set(extract_tail(source_params, config))
        actual = set(archive.files) - {"__metadata__"}
        if actual != expected:
            raise ValueError("Checkpoint tail parameter keys do not match the architecture")
        tail = {key: jax.device_put(archive[key], device) for key in actual}
    classes = metadata.get("classes", [])
    for key in expected:
        shape = ((len(classes), config["hidden_size"]) if key == "classifier.weight"
                 else (len(classes),) if key == "classifier.bias" else source_params[key].shape)
        if tail[key].shape != shape:
            raise ValueError(f"Checkpoint parameter shape mismatch: {key}")
    return {**source_params, **tail}, metadata

### Pillow와 NumPy로 입력 전처리
RGB 이미지를 224×224로 바꾸고 픽셀 값을 `(pixel / 255 - 0.5) / 0.5`로 변환합니다. 모델의 `preprocessor_config.json`과 같은 설정입니다. `preprocess`는 외부 패키지가 아닌 아래 셀에서 직접 정의하는 함수입니다.

In [ ]:
def preprocess(images):
    if not len(images):
        return np.empty((0, 224, 224, 3), dtype=np.float32)
    resized = []
    for pixels in images:
        image = Image.fromarray(np.asarray(pixels, dtype=np.uint8)).convert("RGB")
        image = image.resize((224, 224), Image.Resampling.BILINEAR)
        resized.append(np.asarray(image, dtype=np.float32))
    return (np.stack(resized) / np.float32(255) - np.float32(0.5)) / np.float32(0.5)

## 7. 원본 가중치와 파인튜닝 체크포인트를 직접 읽기
`safetensors`로 원본 가중치를 읽고 앞에서 정의한 `load_checkpoint`로 업데이트한 마지막 블록·분류기를 덮어씁니다. 원본 모델 파일 자체는 변경하지 않습니다.

In [ ]:
from safetensors.numpy import load_file

MODEL_REVISION = "b3428f18dcc7b543470d07f14b4a4157815d1880"
MODEL_SHA256 = "056550dbe6c439dddb35e1800a48aaef86cfdcf8e566ba73f0d1ce41bc7fa1b3"
READY_TO_INFER = IMAGE_PATH.is_file() and CHECKPOINT.is_file()
if READY_TO_INFER:
    weights_path = ROOT / "assets/pretrained/model.safetensors"
    if file_sha256(weights_path) != MODEL_SHA256:
        raise ValueError("원본 가중치 SHA256이 다릅니다.")
    config = json.loads((ROOT / "assets/pretrained/config.json").read_text(encoding="utf-8"))
    source_params = {name: jax.device_put(np.asarray(array, np.float32), device)
                     for name, array in load_file(str(weights_path)).items()}
    params, checkpoint_metadata = load_checkpoint(CHECKPOINT, source_params, config, device)
    print("Checkpoint classes:", checkpoint_metadata["classes"])
else:
    print("IMAGE_PATH와 CHECKPOINT를 실제 파일 경로로 설정하세요. 추론은 아직 실행하지 않습니다.")

## 8. Pillow 전처리 → JAX 추론 → Matplotlib 표시

In [ ]:
if READY_TO_INFER:
    with Image.open(IMAGE_PATH) as image:
        raw = np.asarray(image.convert("RGB"))
    pixels = jax.device_put(preprocess([raw]), device)
    forward = jax.jit(lambda weights, images: original_logits(weights, images, config))
    logits = forward(params, pixels)
    scores = np.asarray(jax.device_get(jax.nn.softmax(logits[0])))
    predicted = int(np.argmax(scores))
    prediction = checkpoint_metadata["classes"][predicted]
    assert len(scores) == len(checkpoint_metadata["classes"]) and np.isfinite(scores).all()
    print("Prediction:", prediction, "softmax:", float(scores[predicted]))
    plt.figure(figsize=(5, 4))
    plt.imshow(raw)
    plt.title(f"{prediction} · softmax {scores[predicted]:.3f}")
    plt.axis("off")
    plt.show()
else:
    print("사진과 체크포인트가 준비된 뒤 이 셀을 다시 실행하세요.")

## 제출할 내용
1. 클래스 구분 기준과 촬영·분할 방식
2. 같은 테스트 이미지에서 분류기 학습과 파인튜닝의 결과
3. 맞힌 사진과 놓친 사진, 다음 실험에서 바꿀 조건 한 가지
4. 실제 GPU·TPU 장치, 결과 회수, 세션 종료 기록

높은 softmax 점수가 물리 동작의 안전성을 보장하지는 않습니다. 여러 물체가 섞인 장면이나 처음 보는 물체에 적용하려면 별도 검증이 필요합니다.